# UJIAN TENGAH SEMESTER — INTELEGENSI BUATAN
**Universitas Teknologi Bandung — Departemen Teknik Informatika (Kelas TIF RP 23D)**

* **Nama:** Anisa Ansuya Henna
* **NIM:** 24552011020
* **Perhitungan Kasus dari NIM:**
  * Jumlah seluruh digit NIM: $2 + 4 + 5 + 5 + 2 + 0 + 1 + 1 + 0 + 2 + 0 = 22$
  * Sisa bagi dengan 10: $22 \bmod 10 = 2$
  * **Kasus Terpilih:** **Kasus 2 — Diagnosis Kanker Payudara** (kasus02_breast_cancer.csv, target: diagnosis)

---
### 6 Tahapan Wajib & Tugas Khusus
1. **Setup & unduh Dataset:**  `kasus02_breast_cancer.csv`
2. **Tahap 1 — Audit data (`fitur.py`)**
3. **Tahap 2 — Model Klasifikasi (`latih.py`):** Membandingkan **Logistic Regression** dan **XGBoost (Gradient Boosting)** menggunakan `Pipeline` `scikit-learn` dan *5-Fold Stratified Cross-Validation* pada data latih.
4. **Tahap 3 — Metrik Evaluasi:** Mengutamakan **Recall** (untuk meminimalkan kesalahan fatal *False Negative* pada diagnosis medis) beserta **ROC-AUC**, **F1-Score**, **Precision**, dan **Accuracy**.
5. **Tahap 4 — Deploy dengan `joblib`:** Menyimpan pipeline terbaik beserta metadata ke `models/model_kasus02.joblib`.
6. **Tahap 5 & 6 — Aplikasi Streamlit (`app.py`) di Google Colab:** Menerima input mentah (manual & CSV), menampilkan prediksi + probabilitas, serta metrik evaluasi.

## Instalasi Library & Pengunduhan Dataset

In [11]:
!pip install streamlit
!npm install -g localtunnel
!mkdir -p dataset models notebook

# Unduh dataset resmi Kasus 2 dari repositori GitHub mata kuliah
!wget -q -O dataset/kasus02_breast_cancer.csv https://raw.githubusercontent.com/Muhammad-Ikhwan-Fathulloh/Artificial-Intelegent-Course-Bank/main/UTS/dataset/kasus02_breast_cancer.csv

import os
import pandas as pd

if os.path.exists("dataset/kasus02_breast_cancer.csv"):
    df_cek = pd.read_csv("dataset/kasus02_breast_cancer.csv")
    print(f"Dataset berhasil dimuat! Ukuran: {df_cek.shape[0]} baris x {df_cek.shape[1]} kolom")
    display(df_cek.head(3))
else:
    print("Gagal mengunduh otomatis. Silakan unggah manual kasus02_breast_cancer.csv ke folder dataset/.")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼Dataset berhasil dimuat! Ukuran: 569 baris x 31 kolom


,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,Ganas
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,Ganas
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,Ganas


## Tahap 1: Audit Data & Feature Engineering (`fitur.py`)

* **Audit data:** Memeriksa jumlah baris/kolom, duplikat, nilai kosong (*missing value*), nilai negatif tidak wajar, dan distribusi kelas target (`Jinak` vs `Ganas`).
* **Pembuatan fitur baru:** Menambahkan 3 fitur rasio (`rasio_worst_mean_radius`, `rasio_worst_mean_texture`, dan `rasio_compactness_smoothness`) untuk menangkap ketidakteraturan ekstrem pada sel tumor.
* **Seleksi fitur multikolinearitas:** Karena fitur geometri seperti `radius`, `perimeter`, dan `area` saling berkorelasi $> 0,98$, fitur dengan korelasi $|r| > 0,92$ dihapus otomatis saat `fit()` dipanggil pada **data latih saja** (mencegah *data leakage*), sekaligus menjaga agar fitur fisik tetap mudah diinterpretasikan secara klinis.

In [12]:
%%writefile fitur.py
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


def audit_data(df: pd.DataFrame) -> None:
    print("=== 1. HASIL AUDIT DATA ===")
    print(f"Ukuran dataset       : {df.shape[0]} baris, {df.shape[1]} kolom")
    print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

    missing = df.isnull().sum()
    missing_ada = missing[missing > 0]
    print(f"Missing value        : {'Tidak ada (0)' if len(missing_ada) == 0 else missing_ada.to_dict()}")

    kolom_numerik = df.select_dtypes(include=[np.number]).columns
    nilai_negatif = (df[kolom_numerik] < 0).sum()
    negatif_ada = nilai_negatif[nilai_negatif > 0]
    print(f"Nilai tidak wajar    : {'Tidak ada nilai negatif' if len(negatif_ada) == 0 else negatif_ada.to_dict()}")

    if "diagnosis" in df.columns:
        print("\nDistribusi Target (diagnosis):")
        print(df["diagnosis"].value_counts())
    print("===========================\n")


def encode_target(y: pd.Series) -> np.ndarray:
    peta_label = {
        "ganas": 1, "m": 1, "malignant": 1, "1": 1,
        "jinak": 0, "b": 0, "benign": 0, "0": 0,
    }
    y_bersih = y.astype(str).str.strip().str.lower().map(peta_label)
    if y_bersih.isnull().any():
        raise ValueError(f"Label target tidak dikenali: {y[y_bersih.isnull()].unique()}")
    return y_bersih.astype(int).values


class RekayasaFiturKanker(BaseEstimator, TransformerMixin):

    def __init__(self, corr_threshold: float = 0.92):
        self.corr_threshold = corr_threshold

    def _tambah_rasio(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

        kolom_buang = [
            c for c in df.columns
            if c in ["id", "unnamed:_0", "unnamed:_32"] or c.startswith("unnamed")
        ]
        if kolom_buang:
            df = df.drop(columns=kolom_buang)

        eps = 1e-6
        if "worst_radius" in df.columns and "mean_radius" in df.columns:
            df["rasio_worst_mean_radius"] = df["worst_radius"] / (df["mean_radius"] + eps)
        if "worst_texture" in df.columns and "mean_texture" in df.columns:
            df["rasio_worst_mean_texture"] = df["worst_texture"] / (df["mean_texture"] + eps)
        if "mean_compactness" in df.columns and "mean_smoothness" in df.columns:
            df["rasio_compactness_smoothness"] = df["mean_compactness"] / (df["mean_smoothness"] + eps)

        return df

    def fit(self, X: pd.DataFrame, y=None):
        df = self._tambah_rasio(X)
        matriks_korelasi = df.corr().abs()
        segitiga_atas = matriks_korelasi.where(
            np.triu(np.ones(matriks_korelasi.shape), k=1).astype(bool)
        )
        self.kolom_dibuang_ = [
            col for col in segitiga_atas.columns if any(segitiga_atas[col] > self.corr_threshold)
        ]
        self.fitur_terpilih_ = [c for c in df.columns if c not in self.kolom_dibuang_]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = self._tambah_rasio(X)
        return df[self.fitur_terpilih_]


Overwriting fitur.py


## Tahap 2, 3, & 4: Pelatihan Model, Evaluasi, & Deploy Joblib (`latih.py`)

Skrip `latih.py` mematuhi seluruh aturan wajib UTS:
* **Split sebelum preprocessing:** Data dibagi 80% *train* dan 20% *test* (`stratify=y`) di paling awal sebelum komputasi statistik apa pun.
* **Perbandingan 2 Algoritma:** Membandingkan **Logistic Regression** (`class_weight="balanced"`) dan **XGBoost / Gradient Boosting** (`scale_pos_weight` dihitung hanya dari `y_train`) menggunakan **5-Fold Stratified Cross-Validation** pada data latih.
* **Evaluasi Satu Kali pada Data Test:** Model terbaik (berdasarkan metrik utama medis **Recall** dan **ROC-AUC**) diuji **satu kali** pada `X_test`.
* **Deploy `joblib`:** Menyimpan pipeline lengkap beserta metadata ke `models/model_kasus02.joblib`.

In [23]:
%%writefile latih.py
from pathlib import Path
import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from fitur import RekayasaFiturKanker, audit_data, encode_target


def main():
    path_dataset = Path("dataset/kasus02_breast_cancer.csv")
    folder_model = Path("models")
    folder_model.mkdir(parents=True, exist_ok=True)
    path_output_model = folder_model / "model_kasus02.joblib"

    if not path_dataset.exists():
        raise FileNotFoundError(
            f"File {path_dataset} tidak ditemukan. Letakkan kasus02_breast_cancer.csv di folder dataset/."
        )

    df = pd.read_csv(path_dataset)
    audit_data(df)
    df = df.drop_duplicates().reset_index(drop=True)

    kolom_buang_awal = [
        c for c in df.columns
        if c.strip().lower() in ["id", "unnamed: 0", "unnamed: 32"] or c.strip().lower().startswith("unnamed")
    ]
    if kolom_buang_awal:
        df = df.drop(columns=kolom_buang_awal)

    X = df.drop(columns=["diagnosis"])
    y = encode_target(df["diagnosis"])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    rasio_kelas_train = float((y_train == 0).sum() / (y_train == 1).sum())
    print(f"Data Train: {len(X_train)} sampel | Data Test: {len(X_test)} sampel")
    print(f"Rasio scale_pos_weight XGBoost (dari data train): {rasio_kelas_train:.4f}\n")

    daftar_pipeline = {
        "Logistic Regression": Pipeline([
            ("fe", RekayasaFiturKanker(corr_threshold=0.92)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]),
        "XGBoost (Gradient Boosting)": Pipeline([
            ("fe", RekayasaFiturKanker(corr_threshold=0.92)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=4,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=rasio_kelas_train,
                eval_metric="logloss",
                random_state=42,
            )),
        ]),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    daftar_metrik = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    hasil_cv = {}

    print("=== 2. PERBANDINGAN ALGORITMA (5-FOLD CV PADA DATA TRAIN) ===")
    for nama_algo, pipe in daftar_pipeline.items():
        skor = cross_validate(pipe, X_train, y_train, cv=cv, scoring=daftar_metrik)
        hasil_cv[nama_algo] = {m: float(skor[f"test_{m}"].mean()) for m in daftar_metrik}
        print(
            f"{nama_algo:28s} -> Recall: {hasil_cv[nama_algo]['recall']:.4f} | "
            f"ROC-AUC: {hasil_cv[nama_algo]['roc_auc']:.4f} | "
            f"F1: {hasil_cv[nama_algo]['f1']:.4f} | "
            f"Accuracy: {hasil_cv[nama_algo]['accuracy']:.4f}"
        )

    model_terbaik_nama = max(
        hasil_cv.keys(), key=lambda k: (hasil_cv[k]["recall"], hasil_cv[k]["roc_auc"])
    )
    pipeline_terbaik = daftar_pipeline[model_terbaik_nama]
    print(f"\nModel Terbaik Terpilih (Prioritas Recall Medis): {model_terbaik_nama}\n")

    pipeline_terbaik.fit(X_train, y_train)
    y_pred = pipeline_terbaik.predict(X_test)
    y_prob = pipeline_terbaik.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred)

    metrik_test = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred)),
        "recall": float(recall_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_prob)),
        "confusion_matrix": cm.tolist(),
    }

    print("=== 3. EVALUASI FINAL DATA TEST (1 KALI EKSEKUSI) ===")
    for k, v in metrik_test.items():
        if k != "confusion_matrix":
            print(f"{k.upper():10s}: {v:.4f}")
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Jinak", "Ganas"]))

    tahap_fe = pipeline_terbaik.named_steps["fe"]
    paket_deploy = {
        "pipeline": pipeline_terbaik,
        "nama_model": model_terbaik_nama,
        "fitur_mentah": X.columns.tolist(),
        "nilai_median_default": X_train.median().to_dict(),
        "kolom_dibuang_korelasi": tahap_fe.kolom_dibuang_,
        "fitur_terpilih": tahap_fe.fitur_terpilih_,
        "metrik_test": metrik_test,
        "hasil_cv": hasil_cv,
    }

    joblib.dump(paket_deploy, path_output_model)
    print(f"\nPipeline model berhasil dideploy ke: {path_output_model}")


if __name__ == "__main__":
    main()

Overwriting latih.py


### Eksekusi Training & Penyimpanan Model (`models/model_kasus02.joblib`)

In [30]:
!python latih.py

=== 1. HASIL AUDIT DATA ===
Ukuran dataset       : 569 baris, 31 kolom
Jumlah baris duplikat: 0
Missing value        : Tidak ada (0)
Nilai tidak wajar    : Tidak ada nilai negatif

Distribusi Target (diagnosis):
diagnosis
Jinak    357
Ganas    212
Name: count, dtype: int64

Data Train: 455 sampel | Data Test: 114 sampel
Rasio scale_pos_weight XGBoost (dari data train): 1.6765

=== 2. PERBANDINGAN ALGORITMA (5-FOLD CV PADA DATA TRAIN) ===
Logistic Regression          -> Recall: 0.9588 | ROC-AUC: 0.9957 | F1: 0.9643 | Accuracy: 0.9736
XGBoost (Gradient Boosting)  -> Recall: 0.9588 | ROC-AUC: 0.9921 | F1: 0.9644 | Accuracy: 0.9736

Model Terbaik Terpilih (Prioritas Recall Medis): Logistic Regression

=== 3. EVALUASI FINAL DATA TEST (1 KALI EKSEKUSI) ===
ACCURACY  : 0.9649
PRECISION : 0.9750
RECALL    : 0.9286
F1        : 0.9512
ROC_AUC   : 0.9970

Confusion Matrix:
 [[71  1]
 [ 3 39]]

Classification Report:
               precision    recall  f1-score   support

       Jinak       0.96  

## Tahap 5: Aplikasi Web Interaktif Streamlit (`app.py`)



In [35]:
%%writefile app.py
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Diagnosis Kanker Payudara - Kasus 2", layout="wide")
PATH_MODEL = Path("models/model_kasus02.joblib")


@st.cache_resource
def muat_model():
    if not PATH_MODEL.exists():
        return None
    return joblib.load(PATH_MODEL)


def main():
    st.title(" Prediksi Diagnosis Kanker Payudara")
    st.caption("UTS Intelegensi Buatan — Kasus 2 (NIM: 24552011020)")

    paket = muat_model()
    if paket is None:
        st.error("Model belum ditemukan. Jalankan `python latih.py` terlebih dahulu.")
        st.stop()

    pipeline = paket["pipeline"]
    fitur_mentah = paket["fitur_mentah"]
    nilai_default = paket["nilai_median_default"]
    metrik_test = paket["metrik_test"]
    hasil_cv = paket["hasil_cv"]

    with st.sidebar:
        st.header("Metrik Evaluasi Model")
        st.write(f"**Algoritma Terpilih:** {paket['nama_model']}")
        st.write(f"**Fitur Mentah → Akhir:** {len(fitur_mentah)} → {len(paket['fitur_terpilih'])} fitur")

        c1, c2 = st.columns(2)
        c1.metric("Recall (Utama)", f"{metrik_test['recall']*100:.2f}%")
        c2.metric("ROC-AUC", f"{metrik_test['roc_auc']:.4f}")
        c1.metric("Accuracy", f"{metrik_test['accuracy']*100:.2f}%")
        c2.metric("F1-Score", f"{metrik_test['f1']*100:.2f}%")
        st.metric("Precision", f"{metrik_test['precision']*100:.2f}%")

        with st.expander("Detail Cross-Validation & Confusion Matrix"):
            st.write("**Rata-rata 5-Fold CV (Data Train):**")
            st.dataframe(pd.DataFrame(hasil_cv).T)
            st.write("**Confusion Matrix (Data Test):**")
            st.dataframe(
                pd.DataFrame(
                    metrik_test["confusion_matrix"],
                    index=["Aktual: Jinak", "Aktual: Ganas"],
                    columns=["Prediksi: Jinak", "Prediksi: Ganas"],
                )
            )

    tab_manual, tab_csv = st.tabs(["Input Manual (30 Fitur Mentah)", "Unggah File CSV"])

    with tab_manual:
        st.info("Nilai awal diisi otomatis dengan nilai median data train. Ubah angka sesuai hasil lab pasien.")
        with st.form("form_input_mentah"):
            t_mean, t_err, t_worst = st.tabs(["Rata-rata (mean)", "Galat (error)", "Terburuk (worst)"])
            input_user = {}

            kelompok_fitur = [
                (t_mean, [f for f in fitur_mentah if f.startswith("mean")]),
                (t_err, [f for f in fitur_mentah if f.endswith("error")]),
                (t_worst, [f for f in fitur_mentah if f.startswith("worst")]),
            ]
            for tab_aktif, daftar_f in kelompok_fitur:
                with tab_aktif:
                    cols = st.columns(3)
                    for i, f_nama in enumerate(daftar_f):
                        with cols[i % 3]:
                            input_user[f_nama] = st.number_input(
                                f_nama, value=float(nilai_default[f_nama]), format="%.5f"
                            )

            tombol_prediksi = st.form_submit_button(" Prediksi Diagnosis", type="primary")

        if tombol_prediksi:
            df_input = pd.DataFrame([input_user])[fitur_mentah]
            hasil_kelas = int(pipeline.predict(df_input)[0])
            prob = pipeline.predict_proba(df_input)[0]
            prob_jinak, prob_ganas = float(prob[0]), float(prob[1])

            st.divider()
            col_hasil, col_p1, col_p2 = st.columns([2, 1, 1])
            with col_hasil:
                if hasil_kelas == 1:
                    st.error("### Hasil Prediksi: GANAS (Malignant)")
                else:
                    st.success("### Hasil Prediksi: JINAK (Benign)")
            col_p1.metric("Probabilitas Ganas", f"{prob_ganas*100:.2f}%")
            col_p2.metric("Probabilitas Jinak", f"{prob_jinak*100:.2f}%")
            st.progress(prob_ganas, text=f"Skor Probabilitas Keganasan: {prob_ganas*100:.2f}%")

    with tab_csv:
        file_csv = st.file_uploader("Unggah file CSV berformat sama dengan kasus02_breast_cancer.csv", type=["csv"])
        if file_csv is not None:
            df_baru = pd.read_csv(file_csv)
            preds = pipeline.predict(df_baru[fitur_mentah])
            probs = pipeline.predict_proba(df_baru[fitur_mentah])

            df_baru["Prediksi_Diagnosis"] = ["Ganas" if p == 1 else "Jinak" for p in preds]
            df_baru["Probabilitas_Ganas (%)"] = (probs[:, 1] * 100).round(2)
            df_baru["Probabilitas_Jinak (%)"] = (probs[:, 0] * 100).round(2)
            st.dataframe(df_baru[["Prediksi_Diagnosis", "Probabilitas_Ganas (%)", "Probabilitas_Jinak (%)"] + fitur_mentah[:4]])


if __name__ == "__main__":
    main()


Overwriting app.py


## Tahap 6: Menjalankan Aplikasi Streamlit di Google Colab



In [36]:
!streamlit run app.py &>/dev/null&

In [37]:
import urllib.request
import time
import subprocess

# 1. Mengambil IP Publik Colab sebagai password (bila diperlukan)
try:
    ip_publik = urllib.request.urlopen("https://api.ipify.org", timeout=5).read().decode("utf-8").strip()
    print("IP Publik Anda (Gunakan ini jika diminta password):", ip_publik)
except Exception as e:
    print("Gagal mengambil IP publik:", e)

# 2. Menjalankan Port Forwarding menggunakan Serveo (Alternatif terbaik dan stabil)
print("\nMenghubungkan ke Serveo.net...")
print("Tunggu beberapa detik hingga URL publik muncul di bawah ini...")

# Menjalankan SSH tunnel secara background dan menyalurkan port 8501 ke publik
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 serveo.net

IP Publik Anda (Gunakan ini jika diminta password): 34.138.234.2

Menghubungkan ke Serveo.net...
Tunggu beberapa detik hingga URL publik muncul di bawah ini...
Forwarding HTTP traffic from https://29cff884b0afd843-34-138-234-2.serveousercontent.com
